# Chores

In [17]:
pip install opencv-python pandas tqdm

Note: you may need to restart the kernel to use updated packages.


In [3]:
import cv2 as cv
import numpy as np
import os
import pandas as pd
import shutil

In [19]:
pip freeze > requirements.txt

Note: you may need to restart the kernel to use updated packages.


In [ ]:
base_path = os.path.join(".", "paris", "data", "img")
dirs = os.listdir(base_path)

for dir in dirs:
  for file in os.listdir(os.path.join(base_path, dir)):
    src = os.path.join(base_path, dir, file)
    print(src)
    dst = base_path
    shutil.move(src, dst)

In [9]:
for dir in dirs:
  os.removedirs(os.path.join(base_path, dir))

In [15]:
from pathlib import Path

def tree(path=Path("."), depth=3, prefix=""):
    if depth < 0:
        return

    entries = sorted(path.iterdir(), key=lambda p: p.name)

    for i, p in enumerate(entries):
        # Skip hidden directories
        if p.is_dir() and p.name.startswith("."):
            continue

        connector = "└── " if i == len(entries) - 1 else "├── "
        print(prefix + connector + p.name)

        if p.is_dir():
            extension = "    " if i == len(entries) - 1 else "│   "
            tree(p, depth - 1, prefix + extension)

tree(Path("."), depth=2)

├── local-desc.ipynb
├── oxford
│   ├── compute_ap.cpp
│   └── data
│       ├── gt
│       └── img
├── paris
│   └── data
│       ├── gt
│       └── img
├── requirements.txt
└── zips
    ├── gt_files_170407.tgz
    ├── oxbuild_images.tgz.zip
    ├── paris_1.tgz
    ├── paris_120310.tgz
    └── paris_2.tgz


# SIFT

## Setup

In [18]:
%load_ext autoreload
%autoreload 2

from src.paths import ensure_dirs, RESULTS_DIR
from src.features import extract_or_load_sift_meanstd
from src.experiments import generate_rankings
from src.evaluation import compile_compute_ap, evaluate_experiment

import pandas as pd

ensure_dirs()

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Feature extraction / load

In [84]:
oxford_ids, oxford_X = extract_or_load_sift_meanstd("oxford")
paris_ids, paris_X = extract_or_load_sift_meanstd("paris")

# oxford_ids, oxford_X = extract_or_load_sift_meanstd("oxford", force=True)
# paris_ids, paris_X = extract_or_load_sift_meanstd("paris", force=True)

Loaded cached oxford: (5063, 256)
Loaded cached paris: (6392, 256)


In [80]:
print(id)
print(to_remove)

6292
C:\Users\iansg\code\cv\P1\paris\data\img\paris_triomphe_000867.jpg


### Removal of corrupt images

In [ ]:
# to_remove = os.path.join(".", "paris", "data", "img", "paris_louvre_000136.jpg")
# to_remove = os.path.join(".", "paris", "data", "img", "paris_louvre_000146.jpg")
# to_remove = os.path.join(".", "paris", "data", "img", "paris_moulinrouge_000422.jpg")
# to_remove = os.path.join(".", "paris", "data", "img", "paris_museedorsay_001059.jpg")
# to_remove = os.path.join(".", "paris", "data", "img", "paris_notredame_000188.jpg")
# to_remove = os.path.join(".", "paris", "data", "img", "paris_pantheon_000284.jpg")
# to_remove = os.path.join(".", "paris", "data", "img", "paris_pantheon_000960.jpg")
# to_remove = os.path.join(".", "paris", "data", "img", "paris_pantheon_000974.jpg")
# to_remove = os.path.join(".", "paris", "data", "img", "paris_pompidou_000195.jpg")
# to_remove = os.path.join(".", "paris", "data", "img", "paris_pompidou_000196.jpg")
# to_remove = os.path.join(".", "paris", "data", "img", "paris_pompidou_000201.jpg")
# to_remove = os.path.join(".", "paris", "data", "img", "paris_pompidou_000467.jpg")
# to_remove = os.path.join(".", "paris", "data", "img", "paris_pompidou_000640.jpg")
# to_remove = os.path.join(".", "paris", "data", "img", "paris_sacrecoeur_000299.jpg")
# to_remove = os.path.join(".", "paris", "data", "img", "paris_sacrecoeur_000330.jpg")
# to_remove = os.path.join(".", "paris", "data", "img", "paris_sacrecoeur_000353.jpg")
# to_remove = os.path.join(".", "paris", "data", "img", "paris_triomphe_000662.jpg")
# to_remove = os.path.join(".", "paris", "data", "img", "paris_triomphe_000833.jpg")
# to_remove = os.path.join(".", "paris", "data", "img", "paris_triomphe_000863.jpg")
# to_remove = os.path.join(".", "paris", "data", "img", "paris_triomphe_000867.jpg")
# os.remove(to_remove)

## Ranking

In [85]:
for dataset, ids, X in [
    ("oxford", oxford_ids, oxford_X),
    ("paris", paris_ids, paris_X),
]:
    for metric in ["cosine", "euclidean"]:
        exp_name, out_dir = generate_rankings(dataset, ids, X, metric)
        print(dataset, exp_name, out_dir)

Ranking oxford/cosine: 100%|██████████| 55/55 [00:00<00:00, 99.98it/s] 


oxford sift_meanstd_cosine C:\Users\iansg\code\cv\P1\results\rankings\oxford\sift_meanstd_cosine


Ranking oxford/euclidean: 100%|██████████| 55/55 [00:00<00:00, 189.94it/s]


oxford sift_meanstd_euclidean C:\Users\iansg\code\cv\P1\results\rankings\oxford\sift_meanstd_euclidean


Ranking paris/cosine: 100%|██████████| 55/55 [00:00<00:00, 85.15it/s]


paris sift_meanstd_cosine C:\Users\iansg\code\cv\P1\results\rankings\paris\sift_meanstd_cosine


Ranking paris/euclidean: 100%|██████████| 55/55 [00:00<00:00, 147.02it/s]

paris sift_meanstd_euclidean C:\Users\iansg\code\cv\P1\results\rankings\paris\sift_meanstd_euclidean


## Evaluation

In [86]:
compute_ap_exe = compile_compute_ap()
compute_ap_exe

WindowsPath('C:/Users/iansg/code/cv/P1/compute_ap')

In [87]:
all_rows = []

for dataset in ["oxford", "paris"]:
    for exp_name in ["sift_meanstd_cosine", "sift_meanstd_euclidean"]:
        df = evaluate_experiment(dataset, exp_name, compute_ap_exe)
        all_rows.append(df)

ap_df = pd.concat(all_rows, ignore_index=True)
ap_df.head()

Evaluating paris/sift_meanstd_euclidean: 100%|██████████| 55/55 [00:00<00:00, 75.11it/s]


,dataset,experiment,query,ap
0,oxford,sift_meanstd_cosine,all_souls_1,0.076743
1,oxford,sift_meanstd_cosine,all_souls_2,0.062501
2,oxford,sift_meanstd_cosine,all_souls_3,0.071666
3,oxford,sift_meanstd_cosine,all_souls_4,0.271301
4,oxford,sift_meanstd_cosine,all_souls_5,0.159365


### Summary

In [88]:
ap_df.to_csv(RESULTS_DIR / "ap_per_query.csv", index=False)

summary = (
    ap_df
    .groupby(["dataset", "experiment"], as_index=False)
    .agg(map=("ap", "mean"), num_queries=("ap", "count"))
    .sort_values("map", ascending=False)
)

summary.to_csv(RESULTS_DIR / "summary_sorted.csv", index=False)
summary

,dataset,experiment,map,num_queries
2,paris,sift_meanstd_cosine,0.172332,55
3,paris,sift_meanstd_euclidean,0.172328,55
0,oxford,sift_meanstd_cosine,0.143783,55
1,oxford,sift_meanstd_euclidean,0.143769,55


### Inspection

In [89]:
from pathlib import Path

sample = next((RESULTS_DIR / "rankings" / "oxford" / "sift_meanstd_cosine").glob("*.txt"))

print(sample)

with open(sample) as f:
    for _ in range(10):
        print(f.readline().strip())

C:\Users\iansg\code\cv\P1\results\rankings\oxford\sift_meanstd_cosine\all_souls_1.txt
christ_church_000757
all_souls_000015
christ_church_000093
oxford_003587
radcliffe_camera_000447
radcliffe_camera_000160
christ_church_000396
radcliffe_camera_000164
radcliffe_camera_000036
magdalen_000995
